# Global Codebase Profile

High-level codebase health dashboard — shape, composition, growth indicators.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset
ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

## 1. Codebase Shape

In [ ]:
print("CODEBASE SHAPE")
print("=" * 70)

try:
    fm = ds.file_metrics
    tm = ds.target_metrics
    el = ds.edge_list
    
    n_targets = len(tm)
    n_files = len(fm)
    total_sloc = fm["code_lines"].sum()
    n_edges = len(el)
    
    print(f"Total CMake targets: {n_targets:,}")
    print(f"Total source files: {n_files:,}")
    print(f"Total SLOC (source lines of code): {total_sloc:,}")
    print(f"Total dependency edges: {n_edges:,}")
    print(f"Average files per target: {n_files / n_targets:.1f}")
    print(f"Average edges per target: {n_edges / n_targets:.1f}")
    
except Exception as e:
    print(f"Error loading codebase shape: {e}")

## 2. Target Type Distribution

In [ ]:
if not ds.has_file("target_metrics"):
    print("target_metrics not available. Skipping target type analysis.")
else:
    tm = ds.target_metrics
    
    print("\nTARGET TYPE DISTRIBUTION")
    print("=" * 70)
    
    type_counts = tm["target_type"].value_counts()
    print(f"\nTarget type counts:")
    for ttype, count in type_counts.items():
        print(f"  {ttype:<30} {count:>6,}")
    
    # Build time by type
    if "total_build_time_ms" in tm.columns:
        build_by_type = tm.groupby("target_type").agg({
            "total_build_time_ms": ["sum", "mean", "count"]
        }).round(2)
        build_by_type.columns = ["Total (ms)", "Mean (ms)", "Count"]
        build_by_type = build_by_type.sort_values("Total (ms)", ascending=False)
        
        print(f"\nBuild time by target type:")
        for idx, row in build_by_type.iterrows():
            print(f"  {idx:<30} Total: {row['Total (ms)']:>12,.0f} ms  Mean: {row['Mean (ms)']:>10,.0f} ms")
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    type_counts.plot(kind="barh", ax=ax1, color="steelblue")
    ax1.set_xlabel("Count")
    ax1.set_title("Target Count by Type")
    ax1.invert_yaxis()
    
    if "total_build_time_ms" in tm.columns:
        build_totals = tm.groupby("target_type")["total_build_time_ms"].sum().sort_values(ascending=True)
        build_totals.plot(kind="barh", ax=ax2, color="coral")
        ax2.set_xlabel("Build Time (ms)")
        ax2.set_title("Total Build Time by Type")
    
    plt.tight_layout()
    plt.show()

## 3. Size Distribution

In [ ]:
if not ds.has_file("target_metrics"):
    print("target_metrics not available. Skipping size distribution analysis.")
else:
    tm = ds.target_metrics
    
    print("\nSIZE DISTRIBUTION")
    print("=" * 70)
    
    # SLOC distribution
    sloc = tm["code_lines_total"]
    print(f"\nCode lines per target:")
    print(f"  Min: {sloc.min():,}")
    print(f"  Q1:  {sloc.quantile(0.25):,.0f}")
    print(f"  Median: {sloc.median():,.0f}")
    print(f"  Q3:  {sloc.quantile(0.75):,.0f}")
    print(f"  Max: {sloc.max():,}")
    print(f"  Mean: {sloc.mean():,.0f}")
    print(f"  Std: {sloc.std():,.0f}")
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax1.hist(sloc, bins=50, edgecolor="black", color="steelblue", alpha=0.7)
    ax1.set_xlabel("Code Lines")
    ax1.set_ylabel("Target Count")
    ax1.set_title("Distribution of Code Lines per Target")
    ax1.set_yscale("log")
    
    # SLOC vs Build time
    if "total_build_time_ms" in tm.columns:
        build_time = tm["total_build_time_ms"].fillna(0)
        ax2.scatter(sloc, build_time, alpha=0.5, s=20)
        ax2.set_xlabel("Code Lines (log scale)")
        ax2.set_ylabel("Build Time (ms)")
        ax2.set_title("Code Lines vs Build Time")
        ax2.set_xscale("log")
        ax2.set_yscale("log")
    
    plt.tight_layout()
    plt.show()

## 4. Preprocessor Health

In [ ]:
if not ds.has_file("file_metrics"):
    print("file_metrics not available. Skipping preprocessor health analysis.")
else:
    fm = ds.file_metrics
    
    print("\nPREPROCESSOR HEALTH")
    print("=" * 70)
    
    if "expansion_ratio" in fm.columns:
        exp_ratio = fm["expansion_ratio"].dropna()
        print(f"\nHeader expansion ratio (preprocessed/source):")
        print(f"  Mean: {exp_ratio.mean():.2f}x")
        print(f"  Median: {exp_ratio.median():.2f}x")
        print(f"  Min: {exp_ratio.min():.2f}x")
        print(f"  Max: {exp_ratio.max():.2f}x")
        print(f"  Std: {exp_ratio.std():.2f}x")
        
        # High expansion files
        high_exp = fm.nlargest(20, "expansion_ratio")[["source_file", "expansion_ratio"]]
        print(f"\nTop 20 files by expansion ratio:")
        for idx, row in high_exp.iterrows():
            print(f"  {row['source_file']:<50} {row['expansion_ratio']:>6.1f}x")
        
        # Visualize
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(exp_ratio, bins=50, edgecolor="black", color="steelblue", alpha=0.7)
        ax.set_xlabel("Expansion Ratio")
        ax.set_ylabel("File Count")
        ax.set_title("Distribution of Header Expansion Ratios")
        plt.tight_layout()
        plt.show()
    
    # Largest preprocessed files
    if "preprocessed_bytes" in fm.columns:
        print(f"\nTop 20 files by preprocessed size:")
        large_preproc = fm.nlargest(20, "preprocessed_bytes")[["source_file", "preprocessed_bytes"]]
        for idx, row in large_preproc.iterrows():
            print(f"  {row['source_file']:<50} {row['preprocessed_bytes']/1024/1024:>8.1f} MB")

## 5. Codegen Footprint

In [ ]:
if not ds.has_file("target_metrics"):
    print("target_metrics not available. Skipping codegen analysis.")
else:
    tm = ds.target_metrics
    
    print("\nCODEGEN FOOTPRINT")
    print("=" * 70)
    
    if "codegen_ratio" in tm.columns:
        has_codegen = (tm["codegen_ratio"] > 0).sum()
        pct_codegen = 100 * has_codegen / len(tm)
        print(f"\nTargets with codegen: {has_codegen} ({pct_codegen:.1f}%)")
        
        codegen_ratio_stats = tm[tm["codegen_ratio"] > 0]["codegen_ratio"]
        print(f"\nCodegen ratio (for targets with codegen):")
        print(f"  Mean: {codegen_ratio_stats.mean():.2%}")
        print(f"  Median: {codegen_ratio_stats.median():.2%}")
        print(f"  Min: {codegen_ratio_stats.min():.2%}")
        print(f"  Max: {codegen_ratio_stats.max():.2%}")
    
    # Build time comparison
    if "codegen_compile_time_sum_ms" in tm.columns and "authored_compile_time_sum_ms" in tm.columns:
        total_codegen_time = tm["codegen_compile_time_sum_ms"].sum()
        total_authored_time = tm["authored_compile_time_sum_ms"].sum()
        
        print(f"\nCompile time breakdown:")
        print(f"  Authored code: {total_authored_time:,.0f} ms ({100*total_authored_time/(total_authored_time+total_codegen_time):.1f}%)")
        print(f"  Generated code: {total_codegen_time:,.0f} ms ({100*total_codegen_time/(total_authored_time+total_codegen_time):.1f}%)")
    
    # Targets with most codegen
    if "codegen_file_count" in tm.columns and "file_count" in tm.columns:
        high_codegen = tm.nlargest(10, "codegen_ratio")[["cmake_target", "codegen_ratio", "codegen_file_count", "file_count"]]
        print(f"\nTop 10 targets by codegen ratio:")
        for idx, row in high_codegen.iterrows():
            print(f"  {row['cmake_target']:<45} {row['codegen_ratio']:>6.1%} ({int(row['codegen_file_count'])}/{int(row['file_count'])} files)")

## 6. Git Activity

In [ ]:
if not ds.has_file("target_metrics"):
    print("target_metrics not available. Skipping git activity analysis.")
else:
    tm = ds.target_metrics
    
    print("\nGIT ACTIVITY")
    print("=" * 70)
    
    if "git_commit_count_total" in tm.columns:
        commits = tm["git_commit_count_total"]
        print(f"\nCommit count per target:")
        print(f"  Mean: {commits.mean():.0f}")
        print(f"  Median: {commits.median():.0f}")
        print(f"  Min: {commits.min():,}")
        print(f"  Max: {commits.max():,}")
        print(f"  Std: {commits.std():.0f}")
        
        # Top churning targets
        active_targets = tm.nlargest(20, "git_commit_count_total")[["cmake_target", "git_commit_count_total"]]
        print(f"\nTop 20 targets by commit count:")
        for idx, row in active_targets.iterrows():
            print(f"  {row['cmake_target']:<50} {int(row['git_commit_count_total']):>6,} commits")
    
    if "git_churn_total" in tm.columns:
        churn = tm["git_churn_total"]
        print(f"\nCode churn (lines added + deleted) per target:")
        print(f"  Mean: {churn.mean():,.0f}")
        print(f"  Median: {churn.median():,.0f}")
        print(f"  Max: {churn.max():,}")
        
        # Top churning targets
        churning = tm.nlargest(20, "git_churn_total")[["cmake_target", "git_churn_total"]]
        print(f"\nTop 20 targets by code churn:")
        for idx, row in churning.iterrows():
            print(f"  {row['cmake_target']:<50} {int(row['git_churn_total']):>8,} lines")
    
    # Visualize
    if "git_commit_count_total" in tm.columns and "git_churn_total" in tm.columns:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        ax1.hist(commits, bins=50, edgecolor="black", color="steelblue", alpha=0.7)
        ax1.set_xlabel("Commit Count")
        ax1.set_ylabel("Target Count")
        ax1.set_title("Distribution of Commit Counts per Target")
        ax1.set_yscale("log")
        
        ax2.scatter(commits, churn, alpha=0.5, s=20)
        ax2.set_xlabel("Commit Count (log scale)")
        ax2.set_ylabel("Code Churn (lines, log scale)")
        ax2.set_title("Commit Count vs Code Churn")
        ax2.set_xscale("log")
        ax2.set_yscale("log")
        
        plt.tight_layout()
        plt.show()

## Summary

In [ ]:
print("\n" + "=" * 70)
print("CODEBASE PROFILE SUMMARY")
print("=" * 70)
print("\nThis notebook provides a high-level overview of the build's codebase.")
print("Key observations:")
print("  - Target type distribution shows composition of the build")
print("  - Size distribution identifies outliers and typical targets")
print("  - Preprocessor metrics highlight header complexity")
print("  - Codegen footprint quantifies generated vs authored code")
print("  - Git activity shows maintenance burden and churn")
print("\nUse these findings to guide optimization priorities in later notebooks.")
print("=" * 70)